# 第114章 特征选择与模型解释

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 29 / 34 步：调优、比较、解释并保存模型**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 模型比较与基线  →  **本章任务：** 特征选择与模型解释  →  **下一步：** 模型保存与批量推理
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：别人给你一堆特征和一份预测任务，哪些变量真正起作用、模型又靠什么做出判断？这两个问题直接决定结果能不能让人信服。这一章把“黑箱”模型拆开看：用置换重要性找出关键特征，用特征选择去掉噪声，再用解释结果判断结论是否站得住。


## 本章目标

学完本章，你将能够：

- **理解**：理解「特征选择与模型解释」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「特征选择与模型解释」的关键输出指标。
- **迁移**：能把「特征选择与模型解释」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：模型给出一个预测，我们常追问“它到底靠哪个特征”？置换重要性把某个特征“打乱”再看模型掉多少分，掉得多说明模型依赖它。但要分清：模型“依赖”一个特征，不等于它真就是导致结果的“原因”。解释工具描述的是模型，不是现实机制。


- 置换重要性衡量打乱特征后的性能下降（打个比方：把一个特征“打乱”后再看模型掉多少分——掉得多说明它重要；但它只说明模型“依赖”它，不等于它真就是导致结果的原因。）
- 单变量选择忽略交互关系
- 相关特征会分摊重要性
- 解释工具描述模型，不证明现实机制


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | `.fit()` | 先明确样本、特征、目标和验证方式，再训练模型。 | 使用训练集计算置换重要性 |
| 模型、公式与诊断 | `pd.DataFrame()`、`importance.head()`、`.sort_values()`、`.round()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 相关变量中只保留排名第一者 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-114 -->
### 数学推导｜置换重要性衡量性能下降

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜记录未破坏数据时的基准分数。** $s_0=s(X,y)$。

**第 2 步｜只打乱特征 $j$，破坏它与目标及其他特征的对应关系。** 第 $r$ 次置换分数为 $s_{jr}$。

**第 3 步｜计算并重复平均性能下降。** 

$$
I_j=\frac1R\sum_{r=1}^{R}(s_0-s_{jr})
$$

同时报告这些下降的标准差，才能看出重要性是否稳定。

**把上面的关系收束为本章计算式：**

$$
I_j=s(X,y)-s(X_{\pi(j)},y)
$$

**符号解释：** $X_{\pi(j)}$ 表示随机打乱第 $j$ 个特征后的数据。

**代码对应：** 在独立验证/测试数据上多次置换，报告重要性均值与波动。

**使用边界：** 相关特征会共享或替代重要性；重要性不是因果效应。


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=103
)
model = RandomForestClassifier(
    n_estimators=250, min_samples_leaf=3, n_jobs=-1, random_state=103
).fit(X_train, y_train)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：回到示例 1 的模型设定。请把 `n_estimators` 从 250 改成 100，再补全下面的代码，用一行断言确认：模型参数确为 100，且 `X_train` 的特征列数与原始特征列数一致。运行后想一想——只改了这一个参数，特征数量和训练数据的形状会发生变化吗？

**自检方式**：先在 exercise 单元格补全并运行；卡住时回看紧邻的示例 1 代码，仍不确定再来对照 hidden solution 隐藏答案。


In [ ]:
try:
    # 请在下方填写代码
    # 任务：把 n_estimators 改为 100，并验证训练特征列数保持为 X.shape[1]。
    from sklearn.ensemble import RandomForestClassifier

    # 填空：把 n_estimators 改成 100（替换 __N__）

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=15,
    scoring="roc_auc",
    random_state=103,
    n_jobs=-1,
)
importance = pd.DataFrame(
    {
        "feature": X.columns,
        "mean": perm.importances_mean,
        "std": perm.importances_std,
    }
).sort_values("mean", ascending=False)
display(importance.head(10).round(4))
print("负重要性数量:", (importance["mean"] < 0).sum())


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：从原始记录到质量报告

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03", "C04"],
        "amount": [120, 80, 80, None, -20],
    }
)
quality = pd.Series(
    {
        "原始行数": len(raw),
        "重复行数": raw.duplicated().sum(),
        "缺失金额": raw["amount"].isna().sum(),
        "非正金额": (raw["amount"] <= 0).sum(),
    }
)
print(quality.to_string())


### 第一个结果怎么读

项目的第一步不是急着画图或建模，而是量化问题规模。质量报告要能回答：问题有多少、影响哪些字段、下一步如何处理。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
clean = raw.drop_duplicates().copy()
clean["amount_valid"] = clean["amount"].where(clean["amount"] > 0)
summary = clean.groupby("customer_id", as_index=False)["amount_valid"].sum(
    min_count=1
)
print("清洗后行数：", len(clean))
print("有效客户数：", summary["amount_valid"].notna().sum())
print(summary)


### 第二个结果怎么读

第二个实验把质量问题转成可追踪的清洗结果。请同时记录删除、保留和缺失处理规则，不能只报告最后的数字。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：重复主键和缺失值怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03"],
        "amount": [120, 80, None, -20],
    }
)
duplicate_keys = raw["customer_id"].duplicated(keep=False)
invalid_amount = raw["amount"].isna() | raw["amount"].le(0)
print("重复主键行：")
print(raw[duplicate_keys])
print("金额异常行：")
print(raw[invalid_amount])
print("先标记问题，再决定保留、合并或回查。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

项目中不能把异常行静默删除。先输出问题记录和数量，再把处理规则写进项目结论。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 使用训练集计算置换重要性
- 相关变量中只保留排名第一者
- 把模型解释写成因果机制
- 忽略解释结果的抽样波动


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 114.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 114.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 114.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

使用置换重要性、单变量特征选择和部分依赖解释模型，并明确解释边界。


### 你已经掌握

- 使用 SelectKBest
- 计算置换重要性
- 理解部分依赖
- 区分预测解释与因果解释


### 需要注意

- 使用训练集计算置换重要性
- 相关变量中只保留排名第一者
- 把模型解释写成因果机制
- 忽略解释结果的抽样波动


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案：把 n_estimators 改成 100
from sklearn.ensemble import RandomForestClassifier

model2 = RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=103,
)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(f_classif, k=8).fit(X_train, y_train)
selected_features = X.columns[selector.get_support()].tolist()
print(selected_features)
